# 11: Statistical Tests (11_Statistical_Tests.ipynb)

**Sprint:** Sprint 2 – Statistics & Mathematics for AI/ML Engineers

## 1. T-Test

**Definition:** Compares the means of two groups to determine if 
they're significantly different.

**Assumptions:** Data is roughly normally distributed; groups have 
similar variance; samples are independent.

**When to use:** Comparing two group means (e.g., before/after, 
control vs treatment) with small-to-moderate sample sizes.

**When not to use:** More than 2 groups (use ANOVA instead); data 
heavily non-normal with small sample size.

**Business Example:** Comparing average sales before and after a 
marketing campaign.

**Interpretation:** A significant p-value (< 0.05) suggests the 
two group means are genuinely different, not due to random chance.

In [1]:
from scipy import stats

before_campaign = [200, 210, 195, 205, 198]
after_campaign = [250, 245, 260, 255, 248]

t_stat, p_value = stats.ttest_ind(before_campaign, after_campaign)
print("T-statistic:", t_stat, "| p-value:", p_value)
print("Significant difference!" if p_value < 0.05 else "No significant difference.")

# Where it's used: ttest_ind() compares the two independent
# groups' means - the very low p-value here confirms sales
# genuinely changed after the campaign, not just by chance.

T-statistic: -13.306157385308737 | p-value: 9.720246784141552e-07
Significant difference!


## 2. Z-Test

**Definition:** Similar to a T-Test, but used when the population 
standard deviation is KNOWN and sample size is large (typically n > 30).

**Assumptions:** Data is normally distributed; population standard 
deviation is known; large sample size.

**When to use:** Large samples with known population variance - 
common in quality control, large-scale A/B tests.

**When not to use:** Small samples or unknown population standard 
deviation (use T-Test instead).

**Business Example:** A factory checks if today's production batch 
weight differs from the known long-term population average.

**Interpretation:** A significant Z-test result means the sample 
mean genuinely differs from the known population mean.

In [2]:
import numpy as np
from scipy import stats

sample = [102, 98, 101, 105, 99, 103, 100, 104]
population_mean = 100
population_std = 5

z_stat = (np.mean(sample) - population_mean) / (population_std / np.sqrt(len(sample)))
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("Z-statistic:", z_stat, "| p-value:", p_value)

# Where it's used: z_stat measures how many standard errors the
# sample mean is from the KNOWN population mean - unlike T-Test,
# we use the known population_std directly rather than estimating
# it from the sample.

Z-statistic: 0.8485281374238571 | p-value: 0.396143909152074


## 3. Chi-Square Test

**Definition:** Tests whether there's a significant association 
between two CATEGORICAL variables (independence test), or whether 
observed frequencies differ from expected ones.

**Assumptions:** Data is categorical (counts/frequencies); 
observations are independent; expected frequency in each category 
should generally be at least 5.

**When to use:** Checking relationships between categorical 
variables - e.g., "does gender affect product preference?"

**When not to use:** Continuous numeric data (not applicable); very 
small expected frequencies in cells.

**Business Example:** Checking if customer satisfaction (Low/Medium/
High) is associated with which store branch they visited.

**Interpretation:** A significant p-value means the two categorical 
variables are NOT independent - there's a real association between them.

In [3]:
from scipy.stats import chi2_contingency
import numpy as np

# Rows: Branch A/B, Columns: Low/Medium/High satisfaction counts
observed = np.array([[20, 30, 50], [40, 35, 25]])

chi2_stat, p_value, dof, expected = chi2_contingency(observed)
print("Chi-Square statistic:", chi2_stat, "| p-value:", p_value)
print("Branch and satisfaction are related!" if p_value < 0.05 else "No significant relationship.")

# Where it's used: chi2_contingency() compares the OBSERVED counts
# against what we'd EXPECT if branch and satisfaction were totally
# unrelated - a low p-value confirms they genuinely ARE related.

Chi-Square statistic: 15.384615384615387 | p-value: 0.0004563239005810305
Branch and satisfaction are related!


## 4. ANOVA (Analysis of Variance)

**Definition:** Compares the means of THREE OR MORE groups at once 
to check if at least one group differs significantly from the others.

**Assumptions:** Data is normally distributed within each group; 
similar variance across groups; independent samples.

**When to use:** Comparing means across 3+ groups (e.g., testing 3 
different marketing strategies at once) - using multiple T-Tests 
instead would inflate error rates.

**When not to use:** Only 2 groups (use T-Test instead); data 
heavily non-normal.

**Business Example:** Comparing average sales across THREE 
different store layouts to find which performs best.

**Interpretation:** A significant p-value means at least one 
group's mean differs from the others - but ANOVA alone doesn't say 
WHICH group; follow-up tests are needed for that.

In [5]:
from scipy import stats

layout_a = [200, 210, 195, 205, 198]
layout_b = [220, 225, 215, 230, 218]
layout_c = [180, 175, 185, 190, 178]

f_stat, p_value = stats.f_oneway(layout_a, layout_b, layout_c)
print("F-statistic:", f_stat, "| p-value:", p_value)
print("At least one layout differs significantly!" if p_value < 0.05 else "No significant difference.")

# Where it's used: f_oneway() compares all THREE layouts' means
# simultaneously in one test - the very low p-value confirms real
# differences exist among the three store layouts.

F-statistic: 56.65722379603394 | p-value: 7.710465748485498e-07
At least one layout differs significantly!


## 5. Mann-Whitney U Test

**Definition:** A non-parametric alternative to the T-Test - 
compares two independent groups WITHOUT assuming normal 
distribution, using ranks instead of raw values.

**Assumptions:** Samples are independent; data is at least ordinal 
(can be ranked) - does NOT require normal distribution.

**When to use:** Comparing two groups when data is skewed, has 
outliers, or is ordinal (like ratings) - safer than T-Test when 
normality is questionable.

**When not to use:** When data IS normally distributed and T-Test 
assumptions are met (T-Test is more statistically powerful in that case).

**Business Example:** Comparing customer satisfaction RATINGS (1-5 
scale, ordinal, not normally distributed) between two service centers.

**Interpretation:** A significant p-value indicates the two 
groups' distributions genuinely differ, without assuming any 
particular shape for the data.

In [7]:
from scipy.stats import mannwhitneyu

center_a_ratings = [3, 4, 2, 5, 3, 4]
center_b_ratings = [5, 4, 5, 4, 5, 3]

u_stat, p_value = mannwhitneyu(center_a_ratings, center_b_ratings)
print("U-statistic:", u_stat, "| p-value:", p_value)

# Where it's used: mannwhitneyu() ranks all values from BOTH groups
# together, then compares rank sums - avoiding any assumption that
# ratings follow a normal distribution, appropriate for this
# ordinal rating data.

U-statistic: 9.5 | p-value: 0.1807619357155913


## 6. Shapiro-Wilk Test

**Definition:** Tests whether a dataset is likely to have come 
from a NORMALLY distributed population - used to CHECK the 
normality assumption before choosing which statistical test to use.

**Assumptions:** Works best with sample sizes between ~3 and 5000; 
no assumptions about the data itself (that's literally what it's testing).

**When to use:** Before deciding whether to use a parametric test 
(like T-Test/ANOVA) or a non-parametric alternative (like Mann-
Whitney) - this test answers "can I assume normality?"

**When not to use:** Very large datasets (thousands+), where even 
tiny, unimportant deviations from normality get flagged as 
significant, making the test overly sensitive.

**Business Example:** Before comparing average delivery times 
between two warehouses using a T-Test, first run Shapiro-Wilk to 
confirm delivery time data is roughly normal.

**Interpretation:** A p-value below 0.05 suggests the data is 
LIKELY NOT normally distributed (reject normality assumption) - a 
p-value above 0.05 means we don't have strong evidence against normality.

In [8]:
from scipy.stats import shapiro

data = [102, 98, 101, 105, 99, 103, 100, 104, 97, 106]

stat, p_value = shapiro(data)
print("Shapiro-Wilk statistic:", stat, "| p-value:", p_value)
print("Data is likely normal." if p_value > 0.05 else "Data is likely NOT normal.")

# Where it's used: shapiro() checks THIS specific sample against
# the normal distribution assumption - a high p-value here supports
# using T-Test/ANOVA/Z-Test safely, since normality isn't rejected.

Shapiro-Wilk statistic: 0.970164611230666 | p-value: 0.8923673075239242
Data is likely normal.
